# Barrido de BioBERT sobre una partición sin fuga

Vuelve a correr el barrido de 24 configuraciones del servidor, pero sobre una
partición que **no** deja la misma ventana de texto en entrenamiento y en prueba.

Sobre los archivos originales, el 73.9 % de los ejemplos de prueba tenía su ventana
ya vista en entrenamiento, y 61 de los 64 artículos de prueba estaban también en
entrenamiento. Por eso el macro-F1 reportado —0.9024 en dev, 0.8721 en test— mide
sobre todo memorización.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno → GPU*.

El barrido es reanudable. Si la sesión se cae, vuelve a correr la última celda y
retoma donde iba.

## 1. Comprobar que hay GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('CUDA disponible:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Sin GPU. Cambia el tipo de entorno de ejecucion.'

## 2. Fijar las versiones

Las del servidor. Importa: el script de entrenamiento usa `evaluation_strategy=`,
que **desapareció en transformers 4.46** (ahora se llama `eval_strategy`), y su
`WeightedTrainer.compute_loss` tiene la firma vieja. Con la versión de Colab de
hoy truena en la primera corrida.

Al terminar hay que **reiniciar el entorno** (Colab lo pide solo).

In [ ]:
!pip install -q "transformers==4.44.2" "datasets==2.20.0" "accelerate==0.33.0" "numpy<2" scikit-learn

# Comprobar de verdad, no confiar en que el pip salio bien. Sin esto, una
# incompatibilidad de versiones hace fallar las 24 corridas una por una: el
# barrido las anota como error y sigue, y media hora despues hay 24 archivos
# .error y ni un solo resultado.
import importlib, sys

for mod, quiero in (('transformers', '4.44.2'), ('datasets', '2.20.0'),
                    ('accelerate', '0.33.0')):
    try:
        m = importlib.import_module(mod)
    except ImportError:
        raise SystemExit('No quedo instalado %s. Reinicia el entorno y vuelve '
                         'a correr esta celda.' % mod)
    hay = getattr(m, '__version__', '?')
    print('%-14s %-10s %s' % (mod, hay, 'ok' if hay == quiero else 'ESPERABA ' + quiero))
    if hay != quiero:
        raise SystemExit('Version equivocada de %s. Si Colab pidio reiniciar '
                         'el entorno, hazlo y vuelve a correr esta celda.' % mod)

import numpy
assert numpy.__version__.startswith('1.'), 'numpy 2.x rompe datasets 2.20'

# La prueba que de verdad importa: que el script del servidor se pueda usar
# con esta version. TrainingArguments perdio evaluation_strategy en la 4.46.
from transformers import TrainingArguments
TrainingArguments(output_dir='/tmp/_p', evaluation_strategy='epoch')
print('\nevaluation_strategy sigue existiendo: el script del servidor corre.')

## 3. Traer los archivos

Sube a tu Drive una carpeta `pseudomonas-trn/entrada` con **cinco archivos**:

- `entity_marked_train.jsonl`, `entity_marked_dev.jsonl`, `entity_marked_test.jsonl`
- `bio_bert_re_finetune.py`
- `particionar.py` y `barrido.py` (de `etapa2/` en el repositorio)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
RAIZ    = '/content/drive/MyDrive/pseudomonas-trn'
ENTRADA = os.path.join(RAIZ, 'entrada')
os.makedirs('/content/trabajo', exist_ok=True)

falta = [f for f in ('entity_marked_train.jsonl', 'entity_marked_dev.jsonl',
                     'entity_marked_test.jsonl', 'bio_bert_re_finetune.py',
                     'particionar.py', 'barrido.py')
         if not os.path.exists(os.path.join(ENTRADA, f))]
if falta:
    raise SystemExit('Faltan en %s: %s' % (ENTRADA, falta))

for f in os.listdir(ENTRADA):
    shutil.copy(os.path.join(ENTRADA, f), '/content/trabajo/')
os.chdir('/content/trabajo')
print('Copiados:', sorted(os.listdir('.')))

## 4. Reparticionar y verificar

Agrupa por **artículo + ventana de texto** y comprueba que no quedó fuga. Si
queda, sale con error y no hay que gastar GPU.

Sale en 1242 / 163 / 157, casi igual que la partición original (1249 / 156 / 157)
y con las mismas proporciones de clase. La comparación es manzana con manzana:
lo único que cambia es la fuga.

In [ ]:
!python particionar.py --por pmid --salida limpia_por_pmid --intentos 120

## 5. El barrido

Las 24 configuraciones exactas de `run_sweep_biobert.sh`, con su mismo orden y su
misma numeración: aquí `run_22` es también `lr3e-5 ep8 bs16 wu0.1`, la que el
servidor reportó como mejor. Así se compara corrida contra corrida, no solo el
mejor de cada barrido.

Cada una guarda su resultado en Drive apenas termina, así que una desconexión
cuesta una corrida, no el barrido.

Los checkpoints van a `/content` (disco local), **nunca a Drive**: son ~433 MB por
época y mandarlos por red tardaría más que entrenar.

Si la sesión se cae, vuelve a correr esta misma celda.

In [ ]:
!python barrido.py \
    --datos      limpia_por_pmid \
    --script     bio_bert_re_finetune.py \
    --trabajo    /content/runs \
    --resultados /content/drive/MyDrive/pseudomonas-trn/barrido_por_pmid

## 6. La tabla

Se puede correr en cualquier momento, incluso a media sesión, para ver lo que va
habiendo. Escribe `resumen.csv` junto a los resultados.

In [ ]:
!python barrido.py --solo_resumen --datos limpia_por_pmid \
    --resultados /content/drive/MyDrive/pseudomonas-trn/barrido_por_pmid

## 7. Opcional: la partición por par, con validación cruzada

Responde una pregunta distinta: **¿generaliza a relaciones que nunca vio?**

No se puede hacer al mismo tiempo que la anterior. Exigir que no se comparta ni
artículo ni par colapsa el corpus en una sola componente con el 89 % de los
ejemplos: es imposible partirlo. Hay que elegir, y por eso van por separado.

Con 5 pliegues son 5 corridas por configuración. Conviene correrlo **solo con la
mejor configuración** que haya salido del paso 5, no con las 24.

In [ ]:
!python particionar.py --por par --salida limpia_por_par --folds 5 --intentos 60

In [ ]:
# Un barrido por pliegue, cada uno con su propia carpeta de resultados: si
# compartieran una, la huella de los datos haria que el segundo se negara a
# correr, que es exactamente lo que debe pasar.
#
# --solo 22 corre UNA configuracion, la ganadora del paso 5. Las 24 en cada
# uno de los 5 pliegues serian 120 corridas y no aportan nada: los
# hiperparametros ya se eligieron.
GANADORA = 22   # ajusta con lo que diga resumen.csv del paso 6

for i in range(5):
    print('\n' + '=' * 60 + '\npliegue %d\n' % i + '=' * 60)
    !python barrido.py \
        --datos      limpia_por_par/fold_{i} \
        --script     bio_bert_re_finetune.py \
        --solo       {GANADORA} \
        --trabajo    /content/runs \
        --resultados /content/drive/MyDrive/pseudomonas-trn/barrido_por_par/fold_{i}

In [ ]:
# El promedio de los cinco pliegues, con su dispersion. La dispersion es la
# mitad util del resultado: dice si la diferencia contra el barrido por
# articulo es senal o es ruido de particion.
import glob, json, statistics

f1s = []
for d in sorted(glob.glob('/content/drive/MyDrive/pseudomonas-trn/barrido_por_par/fold_*')):
    for r in glob.glob(d + '/run_*.json'):
        m = json.load(open(r))
        if m.get('test_macro_f1') is not None:
            f1s.append((d.split('/')[-1], m['test_macro_f1']))

for n, v in sorted(f1s):
    print('%-10s test macro-F1 %.4f' % (n, v))

if len(f1s) >= 2:
    v = [x[1] for x in f1s]
    print('\n%d pliegues: %.4f +- %.4f  (min %.4f, max %.4f)'
          % (len(v), statistics.mean(v), statistics.stdev(v), min(v), max(v)))
elif f1s:
    print('\nSolo %d pliegue: falta correr los demas.' % len(f1s))
else:
    print('\nNo hay resultados todavia.')

## 8. Bajar la mejor corrida

Los pesos se borran al terminar cada configuración para no llenar el disco. Para
quedarte con los de la ganadora, vuelve a correrla sola con `--conservar`
usando los valores que haya dado la tabla.

In [ ]:
# Ajusta los cuatro valores con lo que diga resumen.csv.
# Los demas son los fijos del .sh del servidor: no los cambies o deja de ser
# el mismo experimento.
LR, EPOCHS, BATCH, WARMUP = '3e-5', 8, 16, 0.1

!python bio_bert_re_finetune.py \
    --train_jsonl limpia_por_pmid/entity_marked_train.jsonl \
    --dev_jsonl   limpia_por_pmid/entity_marked_dev.jsonl \
    --test_jsonl  limpia_por_pmid/entity_marked_test.jsonl \
    --labels_json limpia_por_pmid/label_mapping.json \
    --out_dir     /content/mejor \
    --model_name  dmis-lab/biobert-base-cased-v1.1 \
    --batch_size {BATCH} --epochs {EPOCHS} --lr {LR} --warmup_ratio {WARMUP} \
    --weight_decay 0.01 --max_length 512 --seed 42 \
    --use_class_weights --early_stopping --early_stopping_patience 2

# Solo los pesos y el tokenizador; los optimizer.pt son los que hincharon el
# arbol del servidor a 75 GB y no sirven para inferir.
!mkdir -p /content/drive/MyDrive/pseudomonas-trn/mejor_sin_fuga
!cp /content/mejor/pytorch_model.bin /content/mejor/config.json \
    /content/mejor/tokenizer_config.json /content/mejor/special_tokens_map.json \
    /content/mejor/vocab.txt \
    /content/drive/MyDrive/pseudomonas-trn/mejor_sin_fuga/
!ls -la /content/drive/MyDrive/pseudomonas-trn/mejor_sin_fuga/